# 01 — Exploração do HDFS

Notebook destinado à exploração da camada de armazenamento distribuído da pipeline de e-commerce. A análise verifica a estrutura do HDFS, arquivos disponíveis, volume de eventos, distribuição por tipo e organização temporal dos dados.

**Fluxo analisado:** Python Generator → Flume → HDFS → exploração analítica.

## 1. Configuração

Define os caminhos utilizados pela pipeline e os parâmetros de conexão com o HDFS.

In [ ]:
from pathlib import Path
import json
import os
import subprocess

HDFS_URI = os.getenv("HDFS_URI", "hdfs://localhost:9000")
HDFS_RAW = "/data/raw"
HDFS_PROCESSED = "/data/processed"
HDFS_EVENTS = "/data/raw/events"

print(f"HDFS: {HDFS_URI}")
print(f"Raw: {HDFS_RAW}")
print(f"Processed: {HDFS_PROCESSED}")

## 2. Funções auxiliares para consulta ao HDFS

In [ ]:
def hdfs_command(*args):
    command = ["hdfs", "dfs", *args]
    result = subprocess.run(command, capture_output=True, text=True)
    if result.returncode != 0:
        raise RuntimeError(result.stderr.strip() or "Falha ao executar comando HDFS")
    return result.stdout.strip()


def hdfs_exists(path):
    result = subprocess.run(
        ["hdfs", "dfs", "-test", "-e", path],
        capture_output=True,
    )
    return result.returncode == 0


def hdfs_ls(path):
    if not hdfs_exists(path):
        return []
    output = hdfs_command("-ls", "-R", path)
    return output.splitlines() if output else []

## 3. Verificação da estrutura principal

A primeira validação confirma se os diretórios utilizados pelas camadas raw e processed estão disponíveis no HDFS.

In [ ]:
paths = [HDFS_RAW, HDFS_PROCESSED, HDFS_EVENTS]

for path in paths:
    status = "OK" if hdfs_exists(path) else "NÃO ENCONTRADO"
    print(f"{path}: {status}")

## 4. Exploração dos arquivos armazenados

Lista recursivamente os arquivos encontrados na camada raw de eventos.

In [ ]:
raw_files = hdfs_ls(HDFS_RAW)

print(f"Entradas encontradas: {len(raw_files)}")

for entry in raw_files[:50]:
    print(entry)

## 5. Volume armazenado

O comando `-du -h` permite observar o espaço utilizado pelos dados no HDFS.

In [ ]:
if hdfs_exists(HDFS_RAW):
    print(hdfs_command("-du", "-h", HDFS_RAW))
else:
    print("Diretório raw ainda não disponível no HDFS.")

## 6. Leitura dos eventos

Quando os eventos JSONL estiverem disponíveis, esta célula carrega uma amostra diretamente do HDFS para inspeção.

In [ ]:
import pandas as pd

event_files = [
    line.split()[-1]
    for line in raw_files
    if line and not line.startswith("Found") and line.split()[-1].endswith((".json", ".jsonl"))
]

events = []

if event_files:
    sample_file = event_files[0]
    content = hdfs_command("-cat", sample_file)
    for line in content.splitlines()[:1000]:
        try:
            events.append(json.loads(line))
        except json.JSONDecodeError:
            continue

print(f"Eventos carregados: {len(events)}")

if events:
    display(pd.DataFrame(events).head(10))
else:
    print("Nenhum evento JSONL disponível para exploração.")

## 7. Distribuição por tipo de evento

A distribuição permite verificar se o fluxo contém os quatro tipos definidos no domínio: `CLICK`, `CART`, `ORDER` e `DELIVERY`.

In [ ]:
if events:
    df_events = pd.DataFrame(events)
    event_distribution = (
        df_events["event_type"]
        .value_counts()
        .rename_axis("event_type")
        .reset_index(name="total_events")
    )
    display(event_distribution)
else:
    print("Sem dados para calcular a distribuição.")

## 8. Distribuição temporal

Os eventos são analisados pela data e hora do `event_timestamp`, permitindo observar a organização temporal dos dados.

In [ ]:
if events and "event_timestamp" in df_events.columns:
    df_events["event_timestamp"] = pd.to_datetime(
        df_events["event_timestamp"],
        utc=True,
        errors="coerce",
    )
    df_events["event_date"] = df_events["event_timestamp"].dt.date
    df_events["event_hour"] = df_events["event_timestamp"].dt.hour

    temporal_distribution = (
        df_events.groupby(["event_date", "event_hour"])
        .size()
        .reset_index(name="total_events")
        .sort_values(["event_date", "event_hour"])
    )
    display(temporal_distribution)
else:
    print("Sem timestamps disponíveis para análise temporal.")

## 9. Verificação de campos obrigatórios

Valida a presença dos principais atributos definidos no contrato dos eventos.

In [ ]:
required_fields = {
    "event_id",
    "event_type",
    "event_timestamp",
    "ingestion_timestamp",
    "customer_id",
}

if events:
    available_fields = set(df_events.columns)
    missing_fields = sorted(required_fields - available_fields)
    print("Campos obrigatórios:", sorted(required_fields))
    print("Campos ausentes:", missing_fields or "nenhum")
else:
    print("Sem eventos para validar.")

## 10. Conclusão

A exploração desta etapa permite verificar a chegada dos eventos ao HDFS, sua organização física, volume, distribuição por tipo e organização temporal. Esses dados constituem a base histórica utilizada posteriormente pelo processamento batch com Spark e pela consolidação no Hive.